## 1) Setup Environment & Struktur Proyek

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
```

In [9]:
from pathlib import Path

ROOT = Path.cwd()
for folder in [
    ROOT / "data" / "raw",
    ROOT / "data" / "processed",
    ROOT / "models",
    ROOT / "src",
    ROOT / "notebooks",
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Root project:", ROOT)
print("Struktur folder siap.")

Root project: d:\TugasUnud\Semester 6\BIG DATA\Analisis Sentimen\notebooks
Struktur folder siap.


## 2) Install Dependensi dan Konfigurasi API YouTube

```powershell
pip install -r requirements.txt
```

In [10]:
import os
import time
from typing import List, Dict, Optional

import pandas as pd
from dotenv import load_dotenv
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

load_dotenv()

YOUTUBE_API_KEY = os.getenv("YOUTUBE_API_KEY", "").strip()
video_ids_raw = os.getenv("YOUTUBE_VIDEO_IDS", "")
YOUTUBE_VIDEO_IDS = [v.strip() for v in video_ids_raw.split(",") if v.strip()]

max_comments_raw = os.getenv("MAX_COMMENTS", "").strip()
MAX_COMMENTS = int(max_comments_raw) if max_comments_raw else None

print("API key loaded:", bool(YOUTUBE_API_KEY))
print("Jumlah video ID:", len(YOUTUBE_VIDEO_IDS))
print("Max comments per video:", "SEMUA" if MAX_COMMENTS is None else MAX_COMMENTS)

API key loaded: True
Jumlah video ID: 5
Max comments per video: SEMUA


## 3) Ambil Komentar YouTube via YouTube Data API

In [11]:
def fetch_replies(
    youtube,
    video_id: str,
    parent_comment_id: str,
    retry: int = 5,
) -> List[Dict[str, str]]:
    replies: List[Dict[str, str]] = []
    next_page_token = None

    while True:
        res = None
        for attempt in range(retry):
            try:
                req = youtube.comments().list(
                    part="snippet",
                    parentId=parent_comment_id,
                    maxResults=100,
                    pageToken=next_page_token,
                    textFormat="plainText",
                )
                res = req.execute()
                break
            except HttpError as err:
                if attempt == retry - 1:
                    return replies
                time.sleep(min(2 ** (attempt + 1), 30))

        if res is None:
            break

        items = res.get("items", [])
        for item in items:
            sn = item.get("snippet", {})
            replies.append(
                {
                    "video_id": video_id,
                    "comment_id": item.get("id", ""),
                    "parent_comment_id": parent_comment_id,
                    "is_reply": True,
                    "author": sn.get("authorDisplayName", ""),
                    "text": sn.get("textDisplay", ""),
                    "like_count": sn.get("likeCount", 0),
                    "published_at": sn.get("publishedAt", ""),
                }
            )

        next_page_token = res.get("nextPageToken")
        if not next_page_token:
            break

        time.sleep(0.2)

    return replies


def fetch_comments(
    api_key: str,
    video_id: str,
    max_comments: Optional[int] = None,
    retry: int = 5,
    include_replies: bool = True,
) -> List[Dict[str, str]]:
    if not api_key or not video_id:
        raise ValueError("YOUTUBE_API_KEY atau video_id tidak valid")

    youtube = build("youtube", "v3", developerKey=api_key)
    comments: List[Dict[str, str]] = []
    next_page_token = None
    page_count = 0
    seen_comment_ids = set()

    while True:
        res = None
        for attempt in range(retry):
            try:
                req = youtube.commentThreads().list(
                    part="snippet",
                    videoId=video_id,
                    maxResults=100,
                    pageToken=next_page_token,
                    order="time",
                    textFormat="plainText",
                )
                res = req.execute()
                break
            except HttpError as err:
                status = getattr(err.resp, "status", None)
                if status == 403:
                    print(f"  ERROR 403 pada video {video_id}. Cek quota/API access.")
                    return comments
                if attempt == retry - 1:
                    print(f"  Gagal request thread page {page_count + 1} pada video {video_id}.")
                    return comments
                time.sleep(min(2 ** (attempt + 1), 30))

        if res is None:
            break

        items = res.get("items", [])
        if not items:
            break

        page_count += 1

        for item in items:
            top_obj = item.get("snippet", {}).get("topLevelComment", {})
            top_sn = top_obj.get("snippet", {})
            top_id = top_obj.get("id", "")

            if top_id and top_id not in seen_comment_ids:
                comments.append(
                    {
                        "video_id": video_id,
                        "comment_id": top_id,
                        "parent_comment_id": "",
                        "is_reply": False,
                        "author": top_sn.get("authorDisplayName", ""),
                        "text": top_sn.get("textDisplay", ""),
                        "like_count": top_sn.get("likeCount", 0),
                        "published_at": top_sn.get("publishedAt", ""),
                    }
                )
                seen_comment_ids.add(top_id)

            if include_replies and top_id:
                total_reply_count = item.get("snippet", {}).get("totalReplyCount", 0)
                if total_reply_count > 0:
                    replies = fetch_replies(youtube, video_id, top_id, retry=retry)
                    for rp in replies:
                        rp_id = rp.get("comment_id", "")
                        if rp_id and rp_id not in seen_comment_ids:
                            comments.append(rp)
                            seen_comment_ids.add(rp_id)

            if max_comments is not None and len(comments) >= max_comments:
                break

        if max_comments is not None and len(comments) >= max_comments:
            print(f"  Reached max_comments: {len(comments)}")
            break

        next_page_token = res.get("nextPageToken")
        if not next_page_token:
            break

        time.sleep(0.2)

    print(f"  Video {video_id}: selesai, page thread={page_count}, total komentar+balasan={len(comments)}")
    return comments


if not YOUTUBE_VIDEO_IDS:
    raise ValueError("Isi YOUTUBE_VIDEO_IDS di .env. Bisa banyak ID dipisahkan koma.")

all_comments = []
print("=" * 60)
print("MULAI MENGAMBIL KOMENTAR YOUTUBE (TOP-LEVEL + REPLIES)")
print("=" * 60)

for idx, vid in enumerate(YOUTUBE_VIDEO_IDS, 1):
    print(f"\n[{idx}/{len(YOUTUBE_VIDEO_IDS)}] Ambil video: {vid}")
    vid_comments = fetch_comments(
        YOUTUBE_API_KEY,
        vid,
        MAX_COMMENTS,
        retry=5,
        include_replies=True,
    )
    all_comments.extend(vid_comments)
    print(f"           Terambil: {len(vid_comments)}")


df_raw = pd.DataFrame(all_comments)

raw_path_csv = ROOT / "data" / "raw" / "comments_raw_all_videos.csv"
raw_path_parquet = ROOT / "data" / "raw" / "comments_raw_all_videos.parquet"
df_raw.to_csv(raw_path_csv, index=False, encoding="utf-8")
df_raw.to_parquet(raw_path_parquet, index=False)

print("\n" + "=" * 60)
print(f"SELESAI. Total komentar+balasan semua video: {len(df_raw)}")
print("=" * 60)
df_raw.head()

MULAI MENGAMBIL KOMENTAR YOUTUBE (TOP-LEVEL + REPLIES)

[1/5] Ambil video: F6fgLwUeeqI
  Video F6fgLwUeeqI: selesai, page thread=35, total komentar+balasan=4331
           Terambil: 4331

[2/5] Ambil video: MxCqHoldj2Y
  Video MxCqHoldj2Y: selesai, page thread=10, total komentar+balasan=1256
           Terambil: 1256

[3/5] Ambil video: sg8Mzx0fZbU
  Video sg8Mzx0fZbU: selesai, page thread=23, total komentar+balasan=3687
           Terambil: 3687

[4/5] Ambil video: 7CLZkPwhEG4
  Video 7CLZkPwhEG4: selesai, page thread=33, total komentar+balasan=4728
           Terambil: 4728

[5/5] Ambil video: KjXe214MfwQ
  Video KjXe214MfwQ: selesai, page thread=40, total komentar+balasan=5409
           Terambil: 5409

SELESAI. Total komentar+balasan semua video: 19411


,video_id,comment_id,parent_comment_id,is_reply,author,text,like_count,published_at
0,F6fgLwUeeqI,UgzYnpoEjWSG7QJYK6x4AaABAg,,False,@hunink-n8f,SETIAP NEGARA PASTI PUNYA OLIGARKINYA MASING M...,1,2026-01-06T11:03:40Z
1,F6fgLwUeeqI,UgwOAAPFL3rkwPDphHt4AaABAg,,False,@hunink-n8f,TRADISI PEMBODOHAN PADA TUBUH NEGARA YG MADZOR...,1,2026-01-06T11:03:21Z
2,F6fgLwUeeqI,Ugw5kO2JAixOn7m8uhV4AaABAg,,False,@hunink-n8f,JIKA KALIAN INGIN INDONESIA MAJU SILAHKAN BACA...,1,2026-01-06T11:02:53Z
3,F6fgLwUeeqI,UgxvqKSvI6XBfxsyg8t4AaABAg,,False,@indah-n9,Revisi nya kondidera huruf A SDH eror in draft...,0,2025-12-24T13:58:19Z
4,F6fgLwUeeqI,UgxAy8XOeaNVbX7uyst4AaABAg,,False,@aronterbaru2390,EMANG PENDUKUNG NYA PRAVVVOWO ITU PEKOK TUOLOL...,0,2025-06-04T04:03:02Z


In [12]:
print("\n=== DIAGNOSTIK JUMLAH KOMENTAR PER VIDEO ===")

if "is_reply" in df_raw.columns:
    per_video = (
        df_raw.groupby("video_id")
        .agg(
            total=("comment_id", "count"),
            top_level=("is_reply", lambda s: int((~s).sum())),
            replies=("is_reply", lambda s: int(s.sum())),
        )
        .sort_values("total", ascending=False)
    )
    print(per_video)
else:
    comments_per_video = df_raw.groupby("video_id").size()
    for vid, count in comments_per_video.items():
        print(f"{vid}: {count} komentar")

print(f"\nTotal rows: {len(df_raw)}")
if "comment_id" in df_raw.columns:
    print(f"Total unique comment_id: {df_raw['comment_id'].nunique()}")

if "video_id" in df_raw.columns:
    found_ids = sorted(df_raw["video_id"].astype(str).unique().tolist())
else:
    found_ids = []

print(f"\nVideo ID yang berhasil diambil: {found_ids}")
print(f"Video ID yang diminta: {YOUTUBE_VIDEO_IDS}")

missing_vids = [v for v in YOUTUBE_VIDEO_IDS if v not in found_ids]
if missing_vids:
    print("\nVideo yang tidak muncul di hasil:")
    for v in missing_vids:
        print(f"  - {v}")


=== DIAGNOSTIK JUMLAH KOMENTAR PER VIDEO ===
             total  top_level  replies
video_id                              
KjXe214MfwQ   5409       3963     1446
7CLZkPwhEG4   4728       3248     1480
F6fgLwUeeqI   4331       3442      889
sg8Mzx0fZbU   3687       2250     1437
MxCqHoldj2Y   1256        973      283

Total rows: 19411
Total unique comment_id: 19411

Video ID yang berhasil diambil: ['7CLZkPwhEG4', 'F6fgLwUeeqI', 'KjXe214MfwQ', 'MxCqHoldj2Y', 'sg8Mzx0fZbU']
Video ID yang diminta: ['F6fgLwUeeqI', 'MxCqHoldj2Y', 'sg8Mzx0fZbU', '7CLZkPwhEG4', 'KjXe214MfwQ']


## 4) Pembersihan, Normalisasi Menyeluruh, dan Negasi Bertingkat

Tahap ini mencakup normalisasi teks yang lebih komprehensif:
- pembersihan URL/mention/hashtag/emoji/noise
- normalisasi slang Indonesia
- normalisasi karakter berulang
- stopword removal terkontrol
- penanganan negasi bertingkat dengan fitur `NEG_` (contoh: `tidak bagus` -> `NEG_bagus`)
- stemming Sastrawi

Output utama untuk modeling: `text_model`.

In [15]:
import re
import html
import string
import unicodedata
from functools import lru_cache
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

factory = StemmerFactory()
stemmer = factory.create_stemmer()

# Comprehensive slang normalization map (extend as needed)
slang_map = {
    "gk": "tidak", "ga": "tidak", "gak": "tidak", "ngga": "tidak", "nggak": "tidak", "tak": "tidak",
    "bgt": "banget", "bgtt": "banget", "bngt": "banget", "dr": "dari", "drpd": "daripada",
    "yg": "yang", "dgn": "dengan", "krn": "karena", "udh": "sudah", "sm": "sama",
    "tp": "tapi", "tdk": "tidak", "trs": "terus", "aja": "saja", "jg": "juga",
    "blm": "belum", "bkn": "bukan", "org": "orang", "utk": "untuk", "pdhl": "padahal"
}

negator_words = {"tidak", "bukan", "kurang", "belum", "jangan"}

base_stopwords = {
    "yang", "dan", "di", "ke", "dari", "untuk", "dengan", "atau", "itu", "ini", "karena",
    "pada", "jadi", "saya", "aku", "kami", "kita", "kamu", "dia", "mereka", "nya", "sih",
    "nih", "deh", "dong", "lah", "pun", "ya", "iya", "kok", "kan"
}

# Keep negation tokens as signal features.
stopwords_id = base_stopwords - negator_words

invalid_text_markers = {
    "", "[deleted]", "[removed]", "deleted", "removed", "komentar ini telah dihapus"
}


EMOJI_PATTERN = re.compile(
    "["
    "\U0001F600-\U0001F64F"
    "\U0001F300-\U0001F5FF"
    "\U0001F680-\U0001F6FF"
    "\U0001F1E0-\U0001F1FF"
    "\U00002700-\U000027BF"
    "\U000024C2-\U0001F251"
    "]+",
    flags=re.UNICODE,
)

REPEAT_PATTERN = re.compile(r"(.)\1{2,}")

def strip_emoji(text: str) -> str:
    return EMOJI_PATTERN.sub(" ", text)


def normalize_repeat_chars(text: str) -> str:
    # Example: "baguuuus" -> "baguus"
    return REPEAT_PATTERN.sub(r"\1\1", text)


def clean_text(text: str) -> str:
    if text is None:
        return ""

    text = str(text)
    text = html.unescape(text)
    text = unicodedata.normalize("NFKC", text)
    text = text.lower()

    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#\w+", " ", text)
    text = strip_emoji(text)
    text = normalize_repeat_chars(text)

    text = re.sub(r"\d+", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_tokens(tokens):
    normalized = []
    for tok in tokens:
        tok = slang_map.get(tok, tok)
        if len(tok) <= 1:
            continue
        if tok in stopwords_id:
            continue
        normalized.append(tok)
    return normalized


def apply_negation_scope(tokens, window: int = 2):
    # Mark tokens following negators as NEG_<token> within a small window.
    negated_idx = set()
    for i, tok in enumerate(tokens):
        if tok in negator_words:
            seen = 0
            j = i + 1
            while j < len(tokens) and seen < window:
                if tokens[j] in negator_words:
                    break
                if tokens[j] not in stopwords_id:
                    negated_idx.add(j)
                    seen += 1
                j += 1

    out = []
    for i, tok in enumerate(tokens):
        if i in negated_idx:
            out.append(f"NEG_{tok}")
        else:
            out.append(tok)
    return out


@lru_cache(maxsize=50000)
def cached_stem(token: str) -> str:
    if not token:
        return ""
    return stemmer.stem(token)

def stem_tokens(tokens):
    stemmed = []
    for tok in tokens:
        if tok.startswith("NEG_"):
            root = cached_stem(tok[4:])
            if root:
                stemmed.append(f"NEG_{root}")
        else:
            root = cached_stem(tok)
            if root:
                stemmed.append(root)
    return stemmed


def preprocess_pipeline(text: str):
    cleaned = clean_text(text)
    if cleaned in invalid_text_markers:
        return {
            "text_clean": "",
            "text_normalized": "",
            "text_bersih": "",
            "text_model": "",
        }

    tokens = cleaned.split()
    tokens = normalize_tokens(tokens)
    tokens_with_neg = apply_negation_scope(tokens, window=2)
    stemmed_tokens = stem_tokens(tokens_with_neg)

    text_clean = cleaned
    text_normalized = " ".join(tokens_with_neg)
    text_bersih = " ".join(stemmed_tokens)
    text_model = text_bersih

    return {
        "text_clean": text_clean,
        "text_normalized": text_normalized,
        "text_bersih": text_bersih,
        "text_model": text_model,
    }


def preprocess_for_model(text: str) -> str:
    return preprocess_pipeline(text)["text_model"]


# Apply robust preprocessing on fetched comments
work_df = df_raw.copy()
work_df["text"] = work_df["text"].fillna("").astype(str)

# Process each unique raw text only once, then map back to the dataframe.
unique_texts = work_df["text"].unique().tolist()
preprocessed_map = {}
for i, txt in enumerate(unique_texts, 1):
    preprocessed_map[txt] = preprocess_pipeline(txt)
    if i % 1000 == 0:
        print(f"Preprocessed {i}/{len(unique_texts)} unique texts...")

prepped = work_df["text"].map(preprocessed_map)
prepped = pd.DataFrame(list(prepped))
work_df = pd.concat([work_df.reset_index(drop=True), prepped.reset_index(drop=True)], axis=1)

before_rows = len(work_df)
work_df = work_df[work_df["text_model"].str.len() > 0].copy()
after_invalid_filter = len(work_df)

# Deduplicate repeated normalized comments in same video (common spam pattern)
work_df = work_df.drop_duplicates(subset=["video_id", "text_model"], keep="first")
after_dedup = len(work_df)

print(f"Rows awal: {before_rows}")
print(f"Setelah buang text invalid/kosong: {after_invalid_filter}")
print(f"Setelah deduplikasi (video_id + text_model): {after_dedup}")

# Optional quick stats for negation features
neg_count = work_df["text_normalized"].str.contains(r"\bNEG_", regex=True).sum()
print(f"Rows dengan fitur negasi (NEG_): {int(neg_count)}")

df = work_df.reset_index(drop=True)

processed_path = ROOT / "data" / "processed" / "comments_processed.csv"
df.to_csv(processed_path, index=False, encoding="utf-8")

print(f"File tersimpan: {processed_path}")
df[["video_id", "is_reply", "text", "text_normalized", "text_model"]].head(10)

Preprocessed 1000/18987 unique texts...
Preprocessed 2000/18987 unique texts...


KeyboardInterrupt: 

In [ ]:
# Quick QA: cek hasil negasi bertingkat pada data yang sudah dinormalisasi
neg_samples = df[df["text_normalized"].str.contains(r"\bNEG_", regex=True, na=False)][
    ["text", "text_normalized", "text_model"]
].head(15)

print(f"Jumlah baris dengan fitur NEG_: {int(df['text_normalized'].str.contains(r'\\bNEG_', regex=True, na=False).sum())}")
neg_samples

## 5) Pelabelan Sentimen (Rule-based atau Model Pretrained)

In [14]:
import html

positive_words = {
    "bagus", "mantap", "keren", "suka", "terbaik", "hebat", "bantu", "jelas",
    "mantul", "oke", "puas", "berguna", "bermanfaat", "informatif", "mudah",
    "paham", "top", "recommended", "asik", "solusi", "rapi",
    "cepat", "baik", "jujur", "helpful", "nice", "wow"
}

negative_words = {
    "jelek", "buruk", "benci", "parah", "bohong", "sampah", "kecewa", "susah",
    "payah", "bingung", "lambat", "gagal", "ribet", "kacau", "fatal", "aneh",
    "muak", "bosan", "ngaco", "hoax", "fitnah", "jelek",
    "membosankan", "tidak", "paham", "tidak", "membantu", "slow", "stuck", "error"
}

positive_phrases = {
    "sangat membantu": 2.5,
    "terima kasih": 1.6,
    "makasih": 1.4,
    "bagus banget": 2.0,
    "sangat bagus": 2.2,
    "jelas banget": 2.0,
    "penjelasan jelas": 2.0,
    "luar biasa": 2.0,
    "suka banget": 1.8,
    "sangat informatif": 2.0,
    "mudah dipahami": 2.0,
    "berguna banget": 1.8,
    "recommended": 1.5,
    "sangat bermanfaat": 2.0,
    "top banget": 1.5,
}

negative_phrases = {
    "tidak bagus": 2.5,
    "ga bagus": 2.5,
    "gak bagus": 2.5,
    "nggak bagus": 2.5,
    "kurang jelas": 2.5,
    "tidak jelas": 2.5,
    "ga jelas": 2.5,
    "gak jelas": 2.5,
    "nggak jelas": 2.5,
    "tidak membantu": 2.5,
    "bikin bingung": 2.2,
    "susah dipahami": 2.0,
    "jelek banget": 2.2,
    "buruk banget": 2.2,
    "parah banget": 2.5,
    "kecewa banget": 2.0,
    "biasa aja": 0.6,
}

positive_emoji = {"😊": 0.6, "😁": 0.6, "😍": 0.8, "❤️": 0.8, "👍": 0.6, "🙏": 0.4, "🔥": 0.5, "✨": 0.4, "🥰": 0.7}
negative_emoji = {"😡": 0.8, "👎": 0.7, "😭": 0.5, "😢": 0.4, "💩": 1.0, "😤": 0.6, "🤮": 1.0, "😠": 0.7}

def score_text_for_sentiment(raw_text: str, normalized_text: str) -> float:
    raw = html.unescape(str(raw_text)).lower()
    norm = f' {str(normalized_text).lower()} '
    score = 0.0

    for phrase, weight in positive_phrases.items():
        if phrase in raw or phrase in norm:
            score += weight

    for phrase, weight in negative_phrases.items():
        if phrase in raw or phrase in norm:
            score -= weight

    for emoji, weight in positive_emoji.items():
        score += raw.count(emoji) * weight
    for emoji, weight in negative_emoji.items():
        score -= raw.count(emoji) * weight

    tokens = str(normalized_text).split()
    for i, tok in enumerate(tokens):
        base = tok[4:] if tok.startswith("NEG_") else tok
        prev = tokens[i - 1] if i > 0 else ""
        prev2 = tokens[i - 2] if i > 1 else ""

        weight = 1.0
        if prev in {"sangat", "banget", "amat", "sekali", "benar", "bener"} or prev2 in {"sangat", "banget", "amat", "sekali", "benar", "bener"}:
            weight = 1.4

        if tok.startswith("NEG_"):
            if base in positive_words:
                score -= 1.4 * weight
            elif base in negative_words:
                score += 1.0 * weight
            continue

        if base in positive_words:
            score += 1.0 * weight
        elif base in negative_words:
            score -= 1.0 * weight

    return score

def rule_based_label(raw_text: str, normalized_text: str) -> str:
    score = score_text_for_sentiment(raw_text, normalized_text)
    if score >= 1.0:
        return "positif"
    if score <= -1.0:
        return "negatif"
    return "netral"

# Use both raw text and normalized text so emoji + negation can influence labels.
df["sentiment_score"] = df.apply(lambda row: score_text_for_sentiment(row["text"], row["text_normalized"]), axis=1)
df["label"] = df.apply(lambda row: rule_based_label(row["text"], row["text_normalized"]), axis=1)
print(df["label"].value_counts())
print(df[["sentiment_score"]].describe())
df[["text", "text_normalized", "sentiment_score", "label"]].head(10)

NameError: name 'df' is not defined

## 6) Training Baseline Model (Opsional)

In [ ]:
import joblib
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

train_df = df[df["text_model"].str.len() > 0].copy()
X = train_df["text_model"]
y = train_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Use richer TF-IDF config to capture normalization + negation features.
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

clf = LogisticRegression(max_iter=300, class_weight="balanced")
clf.fit(X_train_vec, y_train)

joblib.dump(clf, ROOT / "models" / "sentiment_model.joblib")
joblib.dump(vectorizer, ROOT / "models" / "tfidf_vectorizer.joblib")
print("Model baseline tersimpan di folder models/")

## 7) Evaluasi Model dan Error Analysis

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = clf.predict(X_test_vec)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print(classification_report(y_test, y_pred))

labels = sorted(y.unique().tolist())
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=labels, yticklabels=labels, cmap="Blues")
plt.title("Confusion Matrix")
plt.xlabel("Prediksi")
plt.ylabel("Aktual")
plt.show()

eval_df = pd.DataFrame({"text": X_test.values, "actual": y_test.values, "pred": y_pred})
misclassified = eval_df[eval_df["actual"] != eval_df["pred"]]
misclassified.head(10)

## 8) Prediksi Sentimen pada Komentar Baru

In [ ]:
model = joblib.load(ROOT / "models" / "sentiment_model.joblib")
vec = joblib.load(ROOT / "models" / "tfidf_vectorizer.joblib")


def predict_sentiment(text_list):
    clean_list = [preprocess_for_model(t) for t in text_list]
    X_vec = vec.transform(clean_list)
    labels = model.predict(X_vec)
    probs = model.predict_proba(X_vec)
    classes = model.classes_

    rows = []
    for raw_txt, prep_txt, lbl, pr in zip(text_list, clean_list, labels, probs):
        score_map = {c: float(p) for c, p in zip(classes, pr)}
        rows.append(
            {
                "text": raw_txt,
                "text_model": prep_txt,
                "pred_label": lbl,
                "scores": score_map,
            }
        )
    return pd.DataFrame(rows)

sample_comments = [
    "Videonya tidak bagus, kurang jelas penjelasannya",
    "Kontennya jelek dan bikin bingung",
    "Biasa aja sih, tidak terlalu spesial",
    "Penjelasannya bagus banget dan sangat membantu"
]

predict_sentiment(sample_comments)

## 9) Visualisasi Hasil (Distribusi, WordCloud, Contoh Komentar)

In [ ]:
from wordcloud import WordCloud

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="label", order=df["label"].value_counts().index)
plt.title("Distribusi Label Sentimen")
plt.show()

all_text = " ".join(df["text_bersih"].dropna().astype(str).tolist())
wc = WordCloud(width=1200, height=600, background_color="white").generate(all_text)

plt.figure(figsize=(12, 5))
plt.imshow(wc, interpolation="bilinear")
plt.axis("off")
plt.title("WordCloud Komentar")
plt.show()

print("Contoh komentar paling positif (berdasarkan confidence model):")
pred_train = predict_sentiment(df["text"].head(200).tolist())
pred_train["score_positif"] = pred_train["scores"].apply(lambda x: x.get("positif", 0.0))
pred_train.sort_values("score_positif", ascending=False).head(5)